# Dataset Research

Single notebook for dataset-level analysis: horizon comparison, label profiling,
feature–target intuition, segment-aware checks, and optional raw-data context.

Uses helpers from `src.analysis.io`, `src.analysis.market_context`, and `src.dataset.schema`.

## 1. Setup

In [ ]:
import warnings
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

from src.config.config import fetch_symbol_config
from src.analysis.io import load_dataset_with_meta
from src.analysis.market_context import (
    load_raw_events, parse_trades, replay_book_samples,
    attach_segment_id, label_profile, dataset_summary_row,
)
from src.dataset.schema import FEATURE_COLUMNS

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:.6f}".format)
%matplotlib inline
plt.rcParams["figure.dpi"] = 100

# --- Plotting helpers ---
def _fmt_xaxis(ax, tz_info, tz_name):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=tz_info))
    ax.set_xlabel(f"time ({tz_name})")

def _save_fig(fig, name, output_dir=None):
    if output_dir:
        fig.savefig(output_dir / f"{name}.png", bbox_inches="tight", dpi=150)

def _save_csv(df, name, output_dir=None):
    if output_dir:
        df.to_csv(output_dir / f"{name}.csv")

def _qrange(s, lo_q=0.01, hi_q=0.99):
    lo = s.quantile(lo_q) if lo_q > 0 else 0
    hi = s.quantile(hi_q)
    tag = f"p{lo_q*100:g}\u2013p{hi_q*100:g}"
    return lo, hi, tag

## 2. Configuration

In [ ]:
# --- Primary dataset ---
DATASET_PATH = Path("../data/datasets/binance/BTCUSDT_v2/dataset_i100_tw1000_w600_h1000.parquet")

# --- Horizon sweep: list parquets to compare (set to [] to skip) ---
HORIZON_SWEEP_PATHS = sorted(DATASET_PATH.parent.glob("dataset_i100_tw1000_w600_h*.parquet"))

# --- Raw data (set to None to skip raw sections entirely) ---
# Derived from dataset metadata by default; override here if needed.
RAW_DATA_PATH = None  # auto-detected from metadata in section 3

# --- Reference price for bps conversion (set None to auto-estimate) ---
REF_PRICE = None  # e.g. 100_000 for BTCUSDT

# --- Display ---
TIMEZONE = "UTC"
PLOT_RESAMPLE = "5min"

# --- Output ---
SAVE_OUTPUTS = False
OUTPUT_DIR = None  # set in section 3 after SYMBOL is known

_TZ = ZoneInfo(TIMEZONE)

print(f"Primary dataset: {DATASET_PATH}")
print(f"Horizon sweep:   {[p.name for p in HORIZON_SWEEP_PATHS]}")

## 3. Load primary dataset

In [ ]:
ds, ds_meta = load_dataset_with_meta(DATASET_PATH)

# --- Symbol & config from metadata ---
SYMBOL = ds_meta["symbol"]
cfg = fetch_symbol_config(SYMBOL)
tick_size = float(cfg.tick_size)
step_size = float(cfg.step_size)

# --- Raw data path from metadata (if not overridden in config) ---
if RAW_DATA_PATH is None and "data_path" in ds_meta:
    RAW_DATA_PATH = Path("..") / ds_meta["data_path"]
    print(f"Raw data path (from metadata): {RAW_DATA_PATH}")

# --- Output dir ---
if SAVE_OUTPUTS:
    OUTPUT_DIR = Path(f"../reports/dataset_research/{SYMBOL}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Reference price ---
if REF_PRICE is not None:
    ref_price = REF_PRICE
else:
    ref_price = ds["label"].std() / 0.7e-4
    print(f"REF_PRICE not set, auto-estimated: {ref_price:,.0f} USDT")

interval_ms = int(ds_meta.get("interval_ms", 100))
horizon_ms = ds_meta.get("horizon_ms", "?")

print(f"\nSymbol: {SYMBOL}  tick={cfg.tick_size}  step={cfg.step_size}")
print(f"Rows: {len(ds):,}  |  interval={interval_ms}ms  horizon={horizon_ms}ms")
print(f"Time: {ds.datetime.min()} — {ds.datetime.max()}")
print(f"Ref price: {ref_price:,.2f} USDT")
print(f"\nMetadata:")
for k, v in sorted(ds_meta.items()):
    print(f"  {k}: {v}")

## 4. Horizon sweep

Compare datasets built with different label horizons.  
All share the same raw data and features — only the label changes.

In [ ]:
if not HORIZON_SWEEP_PATHS:
    print("No horizon sweep paths configured — skipping.")
else:
    sweep_rows = []
    for p in HORIZON_SWEEP_PATHS:
        df_h, meta_h = load_dataset_with_meta(p)
        row = dataset_summary_row(df_h, meta_h)
        row.name = p.stem
        sweep_rows.append(row)
    sweep_df = pd.DataFrame(sweep_rows)
    print("=== Horizon Sweep Summary ===")
    with pd.option_context("display.width", 200, "display.max_columns", 20):
        print(sweep_df.to_string())
    _save_csv(sweep_df, "horizon_sweep", OUTPUT_DIR)

## 5. Label profiling

In [ ]:
# --- Label profile ---
lp = label_profile(ds)
print("=== Label Profile ===")
print(lp.to_string())

label = ds["label"]
label_bps = label / ref_price * 1e4
print(f"\nlabel_mean (bps): {label_bps.mean():.4f}")
print(f"label_std  (bps): {label_bps.std():.4f}")
print(f"label_mean_abs (bps): {label_bps.abs().mean():.4f}")

In [ ]:
# --- Label distribution plots ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Histogram (log y)
ax = axes[0]
lo, hi, tag = _qrange(label, 0.01, 0.99)
ax.hist(label, bins=200, range=(lo, hi), edgecolor="none", alpha=0.8)
ax.set_yscale("log")
ax.set_xlabel("label (USDT)")
ax.set_ylabel("count")
ax.set_title(f"Label distribution ({tag}, log scale)")
ax.axvline(0, color="red", ls="--", linewidth=0.8)

# 2. Magnitude buckets (bps)
ax = axes[1]
abs_bps = label_bps.abs()
edges = [0, 0.1, 0.25, 0.5, 1.0]
bucket_labels = ["zero"]
bucket_counts = [int((label == 0).sum())]
for i in range(len(edges) - 1):
    cnt = int(((abs_bps > edges[i]) & (abs_bps <= edges[i+1])).sum())
    bucket_labels.append(f"({edges[i]},{edges[i+1]}]")
    bucket_counts.append(cnt)
bucket_labels.append(f">{edges[-1]}")
bucket_counts.append(int((abs_bps > edges[-1]).sum()))
colors = ["gray"] + ["tab:blue"] * (len(edges) - 1) + ["tab:orange"]
ax.bar(range(len(bucket_labels)), bucket_counts, color=colors, alpha=0.7)
ax.set_xticks(range(len(bucket_labels)))
ax.set_xticklabels(bucket_labels, fontsize=8)
ax.set_yscale("log")
ax.set_xlabel("|label| (bps)")
ax.set_ylabel("count")
for i, cnt in enumerate(bucket_counts):
    if cnt > 0:
        ax.text(i, cnt, f"{cnt/len(label):.1%}", ha="center", va="bottom", fontsize=7)
ax.set_title("Label magnitude buckets (bps)")

# 3. Label over time
ax = axes[2]
lab_rs = ds.set_index("datetime")["label"].resample("5min")
lab_mean_bps = lab_rs.mean() / ref_price * 1e4
lab_std_bps = lab_rs.std() / ref_price * 1e4
ax.plot(lab_mean_bps.index, lab_mean_bps, linewidth=0.8, label="mean")
ax.fill_between(lab_mean_bps.index,
                lab_mean_bps - lab_std_bps, lab_mean_bps + lab_std_bps,
                alpha=0.2, label="mean ± std")
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_ylabel("bps")
ax.set_title("Label mean & std over time (5min, bps)")
ax.legend()
_fmt_xaxis(ax, _TZ, TIMEZONE)

fig.tight_layout()
_save_fig(fig, "labels", OUTPUT_DIR)
plt.show()

## 6. Segment analysis

Detect segments from timestamp gaps (> `interval_ms`).
Large gaps indicate pipeline resets (sequence gap → re-bootstrap → warmup).
Per-segment stats help catch segments that are too short or have unstable labels.

In [ ]:
attach_segment_id(ds, interval_ms)
seg_stats = ds.groupby("segment_id").agg(
    rows=("timestamp", "count"),
    start=("datetime", "min"),
    end=("datetime", "max"),
    label_mean=("label", "mean"),
    label_std=("label", "std"),
    label_zero_pct=("label", lambda x: (x == 0).mean()),
)
seg_stats["duration"] = seg_stats["end"] - seg_stats["start"]

print(f"=== {len(seg_stats)} segments ===")
print(seg_stats.to_string())

# Warn about short segments
short = seg_stats[seg_stats["rows"] < 100]
if len(short):
    print(f"\nWARNING: {len(short)} segments with <100 rows")

## 7. Feature distributions & outliers

In [ ]:
features = [c for c in FEATURE_COLUMNS if c in ds.columns]
missing = set(FEATURE_COLUMNS) - set(features)
if missing:
    print(f"Note: columns not in dataset, skipped: {missing}")

_log_y = {"spread", "delta_midprice", "buy_volume", "sell_volume"}
ncols = min(4, len(features))
nrows = (len(features) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
for ax, col in zip(np.array(axes).flat, features):
    data = ds[col]
    lo, hi, tag = _qrange(data, 0.01, 0.99)
    ax.hist(data, bins=100, range=(lo, hi), edgecolor="none", alpha=0.8)
    if col in _log_y:
        ax.set_yscale("log")
    ax.set_title(f"{col}\n({tag})", fontsize=9)
    ax.axvline(data.median(), color="red", ls="--", linewidth=0.8)
for ax in np.array(axes).flat[len(features):]:
    ax.set_visible(False)
fig.suptitle("Feature distributions (red = median)", fontsize=12)
fig.tight_layout()
_save_fig(fig, "features", OUTPUT_DIR)
plt.show()

print("=== Feature Stats ===")
print(ds[features].describe().round(6).to_string())

In [ ]:
# --- Outlier check ---
outlier_cols = [c for c in ["spread", "delta_midprice", "buy_volume", "sell_volume", "label"]
                if c in ds.columns]
outlier_df = pd.DataFrame({
    "min": ds[outlier_cols].min(),
    "p0.1": ds[outlier_cols].quantile(0.001),
    "p99.9": ds[outlier_cols].quantile(0.999),
    "max": ds[outlier_cols].max(),
    "max/p99.9": (ds[outlier_cols].max() / ds[outlier_cols].quantile(0.999))
                 .replace([np.inf, -np.inf], np.nan),
})
print("=== Outlier Check ===")
print(outlier_df.round(4).to_string())
for col in outlier_cols:
    p999 = ds[col].quantile(0.999)
    col_max = ds[col].max()
    if p999 > 0 and col_max / p999 > 10:
        print(f"  WARNING: {col} max ({col_max:.2f}) is {col_max/p999:.0f}x p99.9 ({p999:.2f})")
    p001 = ds[col].quantile(0.001)
    col_min = ds[col].min()
    if p001 < 0 and col_min / p001 > 10:
        print(f"  WARNING: {col} min ({col_min:.2f}) is {col_min/p001:.0f}x p0.1 ({p001:.2f})")

## 8. Feature ↔ label intuition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation matrix
ax = axes[0]
corr = ds[features + ["label"]].corr()
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(corr.columns, fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature-label correlation matrix")

# IC bar chart
ax = axes[1]
ic = corr["label"].drop("label").sort_values()
colors = ["green" if v > 0 else "red" for v in ic]
ax.barh(ic.index, ic.values, color=colors, alpha=0.7)
ax.set_xlabel("Pearson correlation with label")
ax.set_title("Feature-label IC (full period)")
ax.axvline(0, color="black", linewidth=0.5)

fig.tight_layout()
_save_fig(fig, "feature_label_corr", OUTPUT_DIR)
plt.show()

In [ ]:
# --- Hourly patterns ---
ds["hour"] = ds["datetime"].dt.tz_convert(TIMEZONE).dt.hour
dg = ds.groupby("hour")

ds_hourly = pd.DataFrame({
    "row_count": dg["label"].count(),
    "label_mean (bps)": dg["label"].mean() / ref_price * 1e4,
    "label_std (bps)": dg["label"].std() / ref_price * 1e4,
    "label_nonzero_pct": dg["label"].apply(lambda x: (x != 0).mean()),
    "avg_spread": dg["spread"].mean(),
    "avg_imbalance_1": dg["imbalance_1"].mean(),
})
ds_hourly.index.name = "hour_of_day"
print(f"=== Hourly Stats ({TIMEZONE}) ===")
with pd.option_context("display.max_columns", 15, "display.width", 180):
    print(ds_hourly.to_string())
_save_csv(ds_hourly, "hourly_stats", OUTPUT_DIR)

# IC by hour
print(f"\n=== Feature-label IC by hour ===")
ic_by_hour = ds.groupby("hour").apply(
    lambda g: g[features + ["label"]].corr()["label"].drop("label"),
    include_groups=False,
)
with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 180):
    print(ic_by_hour.to_string())
_save_csv(ic_by_hour, "ic_by_hour", OUTPUT_DIR)

---
## 9. Raw data context (optional)

Lightweight book replay and trade statistics from raw recorded data.
Provides market regime context (midprice, spread, volatility, depth).
Set `RAW_DATA_PATH = None` to skip this section entirely.

In [ ]:
HAS_RAW = RAW_DATA_PATH is not None and Path(RAW_DATA_PATH).exists()
if not HAS_RAW:
    print("No raw data — skipping.")
else:
    raw_df = load_raw_events(RAW_DATA_PATH)
    trades = parse_trades(raw_df)
    print(f"Events: {len(raw_df):,}  |  Trades: {len(trades):,}")
    print(f"Time: {raw_df.datetime.min()} — {raw_df.datetime.max()}")
    print(f"Event types: {raw_df.event_type.value_counts().to_dict()}")

    # Book replay
    book_df = replay_book_samples(RAW_DATA_PATH, cfg, sample_ms=1000)
    book_df["spread_usd"] = book_df["spread_ticks"] * tick_size
    book_df["log_return"] = book_df.groupby("segment_id")["midprice"].transform(
        lambda s: np.log(s / s.shift(1))
    )
    print(f"Book samples: {len(book_df):,}  |  Segments: {book_df.segment_id.nunique()}")

In [ ]:
if HAS_RAW:
    bk = book_df.set_index("datetime")
    fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

    # Midprice
    mid_rs = bk["midprice"].resample(PLOT_RESAMPLE).last()
    axes[0].plot(mid_rs, linewidth=0.7, color="black")
    axes[0].set_ylabel("midprice (USDT)")
    axes[0].set_title(f"{SYMBOL} midprice")

    # Realized volatility
    rv = bk["log_return"].resample("5min").std() * 1e4
    axes[1].plot(rv, linewidth=0.7, color="tab:red")
    axes[1].set_ylabel("RV (bps, 5min std)")
    axes[1].set_title("Realized volatility")

    # Spread
    sp_rs = bk["spread_ticks"].resample(PLOT_RESAMPLE).mean()
    axes[2].plot(sp_rs, linewidth=0.7, color="tab:purple")
    axes[2].set_ylabel("spread (ticks)")
    axes[2].set_title("Spread")
    _fmt_xaxis(axes[2], _TZ, TIMEZONE)

    fig.tight_layout()
    _save_fig(fig, "raw_overview", OUTPUT_DIR)
    plt.show()

    # Summary
    print("=== Raw Data Summary ===")
    summary = {
        "period": f"{book_df.datetime.min()} — {book_df.datetime.max()}",
        "midprice_range": f"{book_df.midprice.min():.2f} — {book_df.midprice.max():.2f}",
        "avg_spread (ticks)": f"{book_df.spread_ticks.mean():.2f}",
        "1-tick spread pct": f"{(book_df.spread_ticks == 1).mean():.1%}",
    }
    if not trades.empty:
        summary["trades"] = f"{len(trades):,}"
        summary["total_notional (USDT)"] = f"{trades.notional.sum():,.0f}"
        summary["buy_pct"] = f"{(trades.side == 'buy').mean():.1%}"
    for k, v in summary.items():
        print(f"  {k}: {v}")

---
## 10. Save summary

In [ ]:
if SAVE_OUTPUTS:
    print(f"Artifacts saved to {OUTPUT_DIR}")
    for f in sorted(OUTPUT_DIR.glob("*")):
        print(f"  {f.name}")
else:
    print("SAVE_OUTPUTS=False — set to True and re-run to save artifacts.")